# Train YOLO11n cho 21 Biển Báo Giao Thông (Xuất ONNX cho Xe Tự Hành)

**Danh sách 21 lớp (10 biển cũ `0..9` + 11 biển mới `10..20`):**
- `0: left` (Chỉ được rẽ trái) | `1: no_left` (Cấm rẽ trái)
- `2: right` (Chỉ được rẽ phải) | `3: no_right` (Cấm rẽ phải)
- `4: straight` (Chỉ được đi thẳng) | `5: stop` (Dừng lại STOP)
- `6: no_straight` (Cấm đi thẳng) | `7: no_entry` (Cấm đi ngược chiều)
- `8: speed_30` (Tốc độ 30 km/h) | `9: speed_60` (Tốc độ 60 km/h)
- `10: no_u_turn` (Cấm quay đầu) | `11: yield` (Nhường đường tại giao lộ)
- `12: intersection` (Giao nhau cùng cấp) | `13: roundabout` (Đi theo vòng xuyến)
- `14: danger_left` (Chỗ ngoặt nguy hiểm trái) | `15: danger_right` (Chỗ ngoặt nguy hiểm phải)
- `16: speed_10` (Tốc độ 10 km/h) | `17: speed_20` (Tốc độ 20 km/h)
- `18: end_speed_limit` (Hết tất cả lệnh cấm) | `19: road_work` (Công trường)
- `20: danger` (Nguy hiểm khác `!`)

**Hướng dẫn nhanh trên Google Colab:**
1. Chọn **Runtime -> Change runtime type -> T4 GPU**.
2. Upload file `data_21BienBao.zip` lên Google Drive (`MyDrive/data_21BienBao.zip`).
3. Chạy lần lượt 5 ô bên dưới. Xong tải file **`DataTrain.onnx`** về chép vào `P2.NhanDienBienBao/`.

In [ ]:
# 1. Kiểm tra GPU và cài đặt Ultralytics + ONNX
!nvidia-smi
!pip install -q ultralytics onnx onnxslim onnxruntime

In [ ]:
# 2. Nạp bộ dữ liệu data_21BienBao.zip (từ Google Drive hoặc upload trực tiếp)
import os, zipfile

USE_GOOGLE_DRIVE = True  # Đổi thành False nếu muốn upload trực tiếp từ máy tính
DRIVE_ZIP_PATH = "/content/drive/MyDrive/data_21BienBao.zip"
DATASET_DIR = "/content/dataset_21BienBao"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    zip_path = DRIVE_ZIP_PATH
else:
    from google.colab import files
    uploaded = files.upload()
    zip_path = next(iter(uploaded.keys()))

assert os.path.exists(zip_path), f"Khong tim thay file ZIP tai: {zip_path}"
os.makedirs(DATASET_DIR, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(DATASET_DIR)

yaml_path = os.path.join(DATASET_DIR, "data.yaml")
with open(yaml_path, "r", encoding="utf-8") as f:
    yaml_text = f.read()
yaml_text = yaml_text.replace("path: .", f"path: {DATASET_DIR}")
with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_text)

print("Da giai nen xong! Noi dung data.yaml:")
print(yaml_text)

In [ ]:
# 3. Train YOLO11n cho 21 biển (QUAN TRỌNG: fliplr=0.0, flipud=0.0 để không lật ngược Trái <-> Phải!)
from ultralytics import YOLO

IMG_SIZE = 416
SAVE_PROJECT = "/content/drive/MyDrive/runs_signs" if os.path.isdir("/content/drive/MyDrive") else "/content/runs_signs"
model = YOLO("yolo11n.pt")

results = model.train(
    data=yaml_path,
    epochs=85,
    imgsz=IMG_SIZE,
    batch=32,
    patience=20,
    device=0,
    workers=2,
    project=SAVE_PROJECT,
    name="yolo11n_21signs",
    exist_ok=True,
    fliplr=0.0,
    flipud=0.0,
    hsv_h=0.015,
    hsv_s=0.60,
    hsv_v=0.50,
    degrees=10.0,
    scale=0.50,
    mosaic=0.80,
    close_mosaic=12,
)

In [ ]:
# 4. Đánh giá độ chính xác mAP50 trên tập validation
best_pt = os.path.join(SAVE_PROJECT, "yolo11n_21signs", "weights", "best.pt")
best_model = YOLO(best_pt)
metrics = best_model.val(data=yaml_path, imgsz=IMG_SIZE)
print(f"mAP50: {metrics.box.map50:.4f} | mAP50-95: {metrics.box.map:.4f}")

In [ ]:
# 5. Xuất sang ONNX (DataTrain.onnx) và tải về máy
import shutil
from google.colab import files

onnx_exported = best_model.export(
    format="onnx",
    imgsz=IMG_SIZE,
    opset=12,
    simplify=True,
    dynamic=False,
)

out_onnx = "/content/DataTrain.onnx"
shutil.copy2(onnx_exported, out_onnx)
print(f"Da xuat thanh cong: {out_onnx} ({os.path.getsize(out_onnx)/1024/1024:.2f} MB)")

if os.path.isdir("/content/drive/MyDrive"):
    shutil.copy2(out_onnx, "/content/drive/MyDrive/DataTrain.onnx")
    print("Da sao luu len Google Drive: MyDrive/DataTrain.onnx")

files.download(out_onnx)